# Calibración del catálogo

El profesor corre esto **una vez** en Colab (T4, `qwen17b`) antes de soltar el catálogo. Comprueba que el paisaje de puntajes tiene la forma que el proyecto necesita: piso muy abajo, ascenso gradual al reparar ranuras, y un techo cerca de 24% al que **random search no llega**.

No es el notebook del curso.

El catálogo está diseñado para que los efectos colaterales se compongan. Cada ranura ataca una **superficie** distinta de la respuesta —`rol` el prefijo, `estrategia` el largo, `formato` la estructura, `estilo` los caracteres, `cierre` el sufijo— y cada superficie se solapa a medias con una vecina. El solape parcial es lo que hace funcionar el ejercicio: con superficies disjuntas el daño se compone pero cada opción sola es leve y el sorteo encuentra configs buenas; con superficies iguales cada opción es letal pero el daño no se suma y no queda gradiente. A la mitad se tienen las dos cosas.

Las opciones dañinas son **directivas sobre la forma literal de la salida**, no consejos de tono: un 1.7B obedece «empezá con "Happy to help!"» y no obedece «sé cálido», porque la restricción explícita de la petición siempre le gana al estilo. Una versión anterior de este catálogo usaba consejos de estilo y el piso se quedó en ~11%.

Objetivos, todos verificados en la celda 8:

| métrica | objetivo |
|---|---|
| piso (`PESADA`) | ≤ 2% |
| techo (`LIMPIA`) | 22–26% |
| mediana del espacio | 1–5% |
| `P(aleatoria ≥ 15%)` | < 1% |
| random search, 15 evals (el del curso) | ≤ 10% |
| random search, 60 evals | ≤ 13% |
| meseta del ascenso por coordenadas | ≥ 22% |

Cómo leer lo que salga:

- Si una opción dañina casi no baja la precisión, el texto no está haciendo efecto y hay que endurecerlo.
- Si una opción deja la precisión en cero, el efecto es demasiado ancho: tiene que apuntar a menos familias.
- Si random search pasa de 13% con 60 evaluaciones, el ejercicio se resuelve sorteando y no hay nada que optimizar.
- Si el ascenso por coordenadas no despega del piso, el piso quedó bajo la resolución del lote: subir `INSTANCIAS`.

La opción limpia **no** está en el mismo índice en todas las ranuras (`LIMPIA` en `oraculo.py`): si lo estuviera, notar el patrón valdría el ejercicio entero en una consulta.


## 1 · Instalar y bajar los 3 archivos  ·  *al terminar, reinicia el entorno de ejecución*

Baja `oraculo.py`, `ayudas.py` y `datos_visibles.json`, más las librerías que usan el modelo y los verificadores. El reinicio es obligatorio: si siguen sin reiniciar, `transformers` a veces queda a medias y `cargar_modelo` falla con un error opaco.


In [ ]:
# bitsandbytes: checkpoints 4-bit (qwen8b, mistral7b). nltk/spacy/emoji/langdetect:
# los usa el verificador de open-instruct, no el notebook.
!pip install -q -U "bitsandbytes>=0.46.1" transformers accelerate \
                   nltk spacy emoji langdetect immutabledict
!python -m spacy download en_core_web_sm -q
!git clone -q https://github.com/allenai/open-instruct

REPO = "https://raw.githubusercontent.com/DanielMelo404/Proyecto-AyD-algoritmos/main"
# --no-cache: Colab a veces reusa un ayudas.py viejo y el alias no existe.
!wget -q --no-cache -O oraculo.py {REPO}/oraculo.py
!wget -q --no-cache -O ayudas.py {REPO}/ayudas.py
!wget -q --no-cache -O datos_visibles.json {REPO}/datos_visibles.json

# Los datos de nltk se bajan a mano: su downloader rechaza el proxy de Colab.
import io
import urllib.request
import zipfile

NLTK_DATA = "https://raw.githubusercontent.com/nltk/nltk_data/gh-pages/packages"
PAQUETES = [
    ("tokenizers", "punkt"),
    ("tokenizers", "punkt_tab"),
    ("taggers", "averaged_perceptron_tagger"),
    ("taggers", "averaged_perceptron_tagger_eng"),
]
for carpeta, nombre in PAQUETES:
    with urllib.request.urlopen(f"{NLTK_DATA}/{carpeta}/{nombre}.zip") as resp:
        zipfile.ZipFile(io.BytesIO(resp.read())).extractall(f"/root/nltk_data/{carpeta}")

print("LISTO  →  Entorno de ejecución ▸ Reiniciar sesión, y sigue en la celda 2")


## 2 · El modelo

Antes: **Entorno de ejecución ▸ Cambiar tipo de entorno de ejecución ▸ T4 GPU**.
Sin GPU, `cargar_modelo` no arranca.

`cargar_modelo` acepta un alias de la tabla o cualquier id público de Hugging Face (`org/nombre`). En `ayudas.py` hay más alias (`qwen8b`, `mistral7b`): caben en T4 en 4-bit, pero cada consulta tarda más.

| Alias | Checkpoint | Tamaño | Notas |
|---|---|---|---|
| `qwen17b` | `Qwen/Qwen3-1.7B` | 1.7B | el de por defecto; el más rápido |
| `ministral3b` | `mistralai/Ministral-3-3B-Instruct-2512-BF16` | 3.8B | fp16, ~7.7 GB de VRAM |
| `llama3b` | `unsloth/Llama-3.2-3B-Instruct` | 3.2B | fp16, ~6.4 GB de VRAM |

Son tres familias distintas (Qwen, Mistral, Llama): sirve para ver si su configuración generaliza o si solo le funciona a un modelo.

El caché guarda el nombre del modelo en la clave, así que cambiar de modelo **no** reusa respuestas del anterior: vuelve a gastar rollouts.

> Usen el repo `-BF16` de Ministral 3. El repo por defecto es FP8 y la T4 no lo soporta.


In [ ]:
from ayudas import cargar_modelo

# Alias → checkpoint. El nombre entra en la clave de caché: si cambian de
# modelo, las respuestas anteriores no se reusan.
#  "qwen17b"     → Qwen/Qwen3-1.7B
#  "ministral3b" → mistralai/Ministral-3-3B-Instruct-2512-BF16
#  "llama3b"     → unsloth/Llama-3.2-3B-Instruct
modelo = cargar_modelo("qwen17b")


## 3 · El oráculo

`dividir` parte `datos_visibles.json` en búsqueda (150) y validación (300). Busquen solo sobre `busqueda`. `validar` mide la config ya elegida sobre instancias que no se usaron al buscar: sirve para ver si generaliza a **instancias** nuevas, no a tipos de restricción nuevos.

Si Colab se desconecta, descomenten las dos líneas de Drive para no perder el caché.


In [ ]:
# Descomenten estas dos líneas si quieren que el caché sobreviva a una
# desconexión: el path de Drive reemplaza cache_oraculo.json local.
# from google.colab import drive
# drive.mount("/content/drive")

from oraculo import Oraculo, CATALOGO, espacio
from ayudas import cargar_datos, dividir

datos = cargar_datos()
busqueda, validacion = dividir(datos)
# 1.7B en fp16 deja VRAM en la T4: un lote más grande genera más prompts a la vez.
oraculo = Oraculo(modelo, busqueda, validacion, lote=16, presupuesto=32_000_000)
# oraculo = Oraculo(modelo, busqueda, validacion, cache="/content/drive/MyDrive/oraculo_cache.json", lote=16, presupuesto=32_000_000)

# 32768 configs (temperatura 0.0). Para incluir 0.3 y 0.7: espacio(TEMPERATURAS)
CONFIGS = espacio()
print(len(CONFIGS), "configuraciones posibles")


## 4 · Barrido por ranura

Para cada ranura se prueban sus **ocho** opciones con las otras cuatro ranuras en su **opción limpia** (`LIMPIA`, que no está en el mismo índice en todas), sobre las mismas 100 instancias del notebook del curso. La precisión y la familia de `violo` más repetida dicen qué efecto colateral está pegando de verdad.

Los resultados quedan en `RESULTADOS`: la celda 6 los reusa sin generar nada.


In [ ]:
from collections import Counter

from oraculo import CATALOGO, LIMPIA, PESADA, RANURAS

# Mismo lote que el notebook del curso: la partición de búsqueda entera, que ya
# viene curada (100 instancias de 2-3 restricciones, todas rompibles por alguna
# ranura y todas alcanzables dentro de max_new_tokens — ver `dividir`).
#
# El barrido son 8 opciones × 5 ranuras = 40 configuraciones × 100 instancias.
# Es la corrida cara del notebook: conviene cronometrar una sola evaluación
# antes de lanzarla entera.
INSTANCIAS = busqueda
SEMILLA = 1
CONSULTAS = {"n": 0}
# (ranura, índice) → Resultado. Lo llena el barrido, lo lee la celda 6.
RESULTADOS = {}


def config_techo():
    """La opción limpia de cada ranura. Es el techo del ejercicio."""
    return dict(LIMPIA, temperatura=0.0)


def config_piso():
    """La opción más dañina de cada ranura. Es el piso."""
    return dict(PESADA, temperatura=0.0)


def medir(config):
    CONSULTAS["n"] += 1
    return oraculo.evaluar(config, INSTANCIAS, semilla=SEMILLA)


def violo_frecuente(trazas):
    """Familia que más se repite en los fallos: es lo que esa opción está rompiendo."""
    conteo = Counter(t["violo"] for t in trazas if t.get("violo"))
    if not conteo:
        return "—"
    familia, n = conteo.most_common(1)[0]
    return f"{familia} ({n})"


print(f"lote: {len(INSTANCIAS)} instancias\n")
print("=== Barrido: una ranura a la vez, las otras en su opción limpia ===")
for ranura in RANURAS:
    print(f"\n## {ranura}")
    for i, texto in enumerate(CATALOGO[ranura]):
        config = config_techo()
        config[ranura] = i
        r = medir(config)
        RESULTADOS[(ranura, i)] = r
        marca = "  ← limpia" if i == LIMPIA[ranura] else ""
        primera = texto.splitlines()[0]
        print(f"  [{i}] {r.precision:6.1%}  violo: {violo_frecuente(r.trazas)}{marca}")
        print(f"       {primera[:90]}")


## 5 · Extremos

`PESADA` (la opción más dañina de cada ranura) es el piso; `LIMPIA` es el techo. Cada superficie cubre ~50% del lote y las cinco juntas llegan al 96%, así que el piso tiene que quedar muy abajo.

| | objetivo |
|---|---|
| piso (`PESADA`) | ≤ 2% |
| techo (`LIMPIA`) | 22–26% |

Si el piso se queda arriba de 2%, las opciones dañinas no están pegando y hay que endurecer su texto. Si el techo no llega a 22%, las opciones limpias no alcanzan y el ejercicio se queda sin margen.


In [ ]:
piso = config_piso()
techo = config_techo()

r_piso = medir(piso)
r_techo = medir(techo)
print(f"piso   (PESADA): {r_piso.precision:.1%}   objetivo ≤ 2%")
print(f"techo  (LIMPIA): {r_techo.precision:.1%}   objetivo 22–26%")
print("piso ", {r: piso[r] for r in RANURAS})
print("techo", {r: techo[r] for r in RANURAS})


## 6 · Daño medido y paisaje predicho

La masa de instancias en riesgo que se calcula mirando las familias es una **cota superior**: dice a cuántas instancias *podría* pegarle una opción, no si el modelo de verdad hace lo que el texto le sugiere. Esta celda lo mide.

1. **Daño por opción.** Para cada una de las 15 opciones dañinas, qué instancias resolvía el techo y esta opción rompe. Sale del `RESULTADOS` del barrido: **no genera nada nuevo**.
2. **Paisaje predicho.** Uniendo esos conjuntos medidos se predice la precisión de una muestra grande del espacio (32 768 configs) sin evaluarlas. De ahí salen la mediana, `P(aleatoria ≥ 15%)` y la curva de random search.
3. **Contraste.** Se evalúan 8 configs al azar y se compara predicho contra medido. Es lo único que cuesta rollouts (8 × 100).

Las familias que imprime la parte 1 son exactamente lo que va en `SUPERFICIE` y `PESO` del notebook de backtracking.

Objetivo del paisaje:

| métrica | objetivo |
|---|---|
| mediana del espacio | 1–5% |
| `P(aleatoria ≥ 15%)` | < 1% |
| random search, 15 evals (el del curso) | ≤ 10% |
| random search, 60 evals | ≤ 13% |

Si random search pasa de 13% con 60 evaluaciones, el ejercicio se resuelve sorteando y no hay nada que optimizar: hay que endurecer las opciones dañinas.


In [ ]:
import itertools
import random
import statistics

from collections import Counter

# Instancias que el techo resuelve. Todo el paisaje vive acá adentro: ninguna
# opción puede hacer que el modelo acierte una que el techo ya fallaba.
fallo_techo = {t["id"] for t in r_techo.trazas}
resuelve_techo = [x["id"] for x in INSTANCIAS if x["id"] not in fallo_techo]


def rotas_por(ranura, indice):
    """Instancias que el techo resolvía y esta opción rompe.

    Se lee del barrido (`RESULTADOS`), donde esa opción ya se midió con las
    otras cuatro ranuras limpias. La opción limpia no rompe nada por
    definición: devuelve el conjunto vacío.
    """
    if indice == LIMPIA[ranura]:
        return set()
    fallan = {t["id"] for t in RESULTADOS[(ranura, indice)].trazas}
    return fallan & set(resuelve_techo)


# ─── 1 · daño medido, opción por opción ──────────────────────────────
DANO = {}
print("=== Daño medido por opción (las otras cuatro ranuras, limpias) ===")
print(f"el techo resuelve {len(resuelve_techo)}/{len(INSTANCIAS)} instancias\n")
for ranura in RANURAS:
    print(f"## {ranura}")
    for i in range(len(CATALOGO[ranura])):
        if i == LIMPIA[ranura]:
            print(f"  [{i}]   —     limpia")
            continue
        rotas = rotas_por(ranura, i)
        DANO[(ranura, i)] = rotas
        familias = Counter(
            t["violo"] for t in RESULTADOS[(ranura, i)].trazas
            if t["id"] in rotas and t.get("violo")
        )
        top = ", ".join(f"{f}({n})" for f, n in familias.most_common(4)) or "—"
        print(f"  [{i}] {len(rotas) / len(resuelve_techo):5.0%} del techo  ({len(rotas):3} inst)")
        print(f"       {top}")
    print()


# ─── 2 · paisaje predicho sobre todo el espacio ──────────────────────
def predecir(config):
    """Precisión estimada: lo que el techo resuelve, menos lo que rompe cada ranura."""
    rotas = set()
    for ranura in RANURAS:
        rotas |= DANO.get((ranura, config[ranura]), set())
    return (len(resuelve_techo) - len(rotas)) / len(INSTANCIAS)


TODAS = [
    dict(zip(RANURAS, combo), temperatura=0.0)
    for combo in itertools.product(*(range(len(CATALOGO[r])) for r in RANURAS))
]
predichas = sorted(predecir(c) for c in TODAS)

print(f"=== Paisaje predicho ({len(TODAS)} configs, sin generar nada) ===")
print(f"  piso    {predecir(config_piso()):6.1%}   (medido {r_piso.precision:.1%})")
print(f"  techo   {predecir(config_techo()):6.1%}   (medido {r_techo.precision:.1%})")
print(f"  mediana {statistics.median(predichas):6.1%}   objetivo 1–5%")
for umbral in (0.10, 0.15, 0.20):
    frac = sum(1 for p in predichas if p >= umbral) / len(predichas)
    marca = "  ← objetivo < 1%" if umbral == 0.15 else ""
    print(f"  P(aleatoria ≥ {umbral:.0%}) = {frac:6.2%}{marca}")

print("\n=== Random search sobre el paisaje predicho (500 corridas) ===")
for k in (15, 25, 60, 120):
    rng = random.Random(k)
    mejores = sorted(
        max(predecir(rng.choice(TODAS)) for _ in range(k)) for _ in range(500)
    )
    marca = "  ← objetivo ≤ 15%" if k == 60 else ""
    print(
        f"  {k:3} evals: medio {statistics.mean(mejores):5.1%}  "
        f"p50 {mejores[250]:5.1%}  p90 {mejores[450]:5.1%}{marca}"
    )


# ─── 3 · contraste: 8 configs al azar, predicho vs medido ────────────
print("\n=== Contraste (8 configs al azar) ===")
print(f"  {'config':<26} {'predicho':>9} {'medido':>8} {'error':>7}")
errores = []
for config in random.Random(7).sample(TODAS, 8):
    p = predecir(config)
    m = medir(config).precision
    errores.append(abs(p - m))
    indices = "".join(str(config[r]) for r in RANURAS)
    print(f"  {indices:<26} {p:9.1%} {m:8.1%} {p - m:+7.1%}")
print(f"\n  error absoluto medio: {statistics.mean(errores):.1%}")
print("  bajo ~3% el modelo aditivo sirve para predecir el paisaje;")
print("  muy por encima significa que las ranuras interactúan y hay que medir más.")


## 7 · Ascenso por coordenadas

Arranca en `PESADA` (el piso). Ranura por ranura prueba los cuatro índices y se queda con el que más sube. Otra vuelta solo si alguna ranura todavía mejora. La curva es la precisión después de cada cambio.

Esto es lo que random search **no** puede hacer: usar la estructura. Gasta ~46 evaluaciones y debería llegar a 28–30%, muy por encima del ~15% que saca el sorteo con el mismo presupuesto. Que llegue al óptimo está bien — es una heurística informada, y de eso se trata el ejercicio.


In [ ]:
from ayudas import curva


def ascenso():
    """Sube una ranura a la vez desde el piso (`PESADA`).

    En cada ranura se prueban los otros índices y se adopta el mejor.
    Una vuelta que no cambia ninguna ranura es la meseta.
    """
    config = config_piso()
    gastadas = 0
    r = medir(config)
    gastadas += 1
    puntos = [r.precision]
    inicial = {ranura: config[ranura] for ranura in RANURAS}
    print(f"inicio  {r.precision:6.1%}  {inicial}")
    while True:
        mejoro = False
        for ranura in RANURAS:
            mejor_i = config[ranura]
            mejor_p = puntos[-1]
            for i in range(len(CATALOGO[ranura])):
                if i == config[ranura]:
                    continue
                prueba = dict(config)
                prueba[ranura] = i
                rr = medir(prueba)
                gastadas += 1
                if rr.precision > mejor_p:
                    mejor_p = rr.precision
                    mejor_i = i
            if mejor_i != config[ranura]:
                config[ranura] = mejor_i
                puntos.append(mejor_p)
                mejoro = True
                print(f"  {ranura} → {mejor_i}  {mejor_p:6.1%}")
        if not mejoro:
            print("meseta: una vuelta completa sin subida")
            break
    return config, puntos, gastadas


config_greedy, curva_greedy, evals_greedy = ascenso()
optimo = config_techo()
print()
print("meseta en", {ranura: config_greedy[ranura] for ranura in RANURAS}, f"{curva_greedy[-1]:.1%}")
print(
    "óptimo (LIMPIA):",
    {ranura: optimo[ranura] for ranura in RANURAS},
    f"{r_techo.precision:.1%}",
)
print("¿llegó al óptimo?", "sí" if config_greedy == optimo else "no")
curva(curva_greedy)


## 8 · Resumen

Todo junto contra los objetivos. El caché no vuelve a generar lo ya medido en el barrido; el conteo igual anota cada llamada a `evaluar`.

Cómo leer un fallo:

- **Piso arriba de 2%** → alguna opción dañina no está pegando: endurecer su texto para que toque dos grupos de familias, no uno.
- **Techo debajo de 22%** → las opciones limpias no alcanzan; el margen del ejercicio se achicó.
- **`P(aleatoria ≥ 15%)` arriba de 1%** → hay demasiadas configs buenas y el sorteo las encuentra. Suele ser una opción "leve" que salió casi gratis.
- **El ascenso no despega** → el piso quedó por debajo de la resolución del lote (1% con 100 instancias): subir `INSTANCIAS`.

Es un lazo: calibrar → ajustar el texto de la opción que falló → recalibrar. El caché hace que solo se paguen las configs que tocan la opción cambiada.


In [ ]:
def veredicto(valor, minimo=None, maximo=None):
    ok = (minimo is None or valor >= minimo) and (maximo is None or valor <= maximo)
    return "ok" if ok else "REVISAR"


mediana = statistics.median(predichas)
sobre_15 = sum(1 for p in predichas if p >= 0.15) / len(predichas)
rng = random.Random(60)
rs60 = statistics.mean(
    max(predecir(rng.choice(TODAS)) for _ in range(60)) for _ in range(500)
)

filas = [
    ("piso (PESADA)", f"{r_piso.precision:.1%}", "≤ 2%", veredicto(r_piso.precision, maximo=0.02)),
    ("techo (LIMPIA)", f"{r_techo.precision:.1%}", "22–26%", veredicto(r_techo.precision, 0.22, 0.26)),
    ("mediana del espacio", f"{mediana:.1%}", "1–5%", veredicto(mediana, 0.01, 0.05)),
    ("P(aleatoria ≥ 15%)", f"{sobre_15:.2%}", "< 1%", veredicto(sobre_15, maximo=0.01)),
    ("random search, 60 evals", f"{rs60:.1%}", "≤ 13%", veredicto(rs60, maximo=0.13)),
    # Llegar al techo está bien: es una heurística informada, no un sorteo.
    ("meseta greedy", f"{curva_greedy[-1]:.1%}", "≥ 22%", veredicto(curva_greedy[-1], minimo=0.22)),
    ("evaluaciones greedy", str(evals_greedy), "—", ""),
    ("evaluaciones del notebook", str(CONSULTAS["n"]), "—", ""),
    ("llegó al óptimo", "sí" if config_greedy == optimo else "no", "—", ""),
]
ancho = max(len(n) for n, _, _, _ in filas)
print(f"{'métrica':<{ancho}}  {'valor':>8}  {'objetivo':>9}  veredicto")
for nombre, valor, objetivo, v in filas:
    print(f"{nombre:<{ancho}}  {valor:>8}  {objetivo:>9}  {v}")
print("\nconfig greedy", {ranura: config_greedy[ranura] for ranura in RANURAS})
print("config óptima", {ranura: optimo[ranura] for ranura in RANURAS})
